# Recomendacion basada en embeddings semanticos

Este notebook transforma el experimento de clasificacion textual en un sistema sencillo de recomendacion basada en contenido. En lugar de entrenar un clasificador supervisado, aqui cada publicacion se representa mediante un embedding semantico y las recomendaciones se obtienen comparando la cercania entre el perfil de un usuario y los posts disponibles.

La idea central es reutilizar el mismo dataset textual del notebook anterior, mantener la combinacion de `descripcion + hashtags` ya procesada y sustituir la logica de prediccion por una logica de similitud.

In [1]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

try:
    from sentence_transformers import SentenceTransformer
except ImportError as exc:
    raise ImportError(
        "Falta instalar sentence-transformers. Ejecuta: %pip install -q sentence-transformers"
    ) from exc

DATA_PATH = Path("data/posts.csv")
EXPECTED_COLUMNS = ["id", "descripcion", "hashtags", "categoria"]
EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"
TOP_K = 5
INTERESES_USUARIO = ["playa", "viajes"]


c:\Users\rafae\Desktop\TFM2\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Carga del dataset textual

Se utiliza el mismo archivo `data/posts.csv` del notebook anterior. Igual que en el experimento de clasificacion, la señal textual principal se construye uniendo `descripcion` y `hashtags`.

In [2]:
df = pd.read_csv(DATA_PATH, encoding="utf-8")

actual_columns = list(df.columns)
if set(actual_columns) != set(EXPECTED_COLUMNS):
    raise ValueError(
        "El CSV no tiene el esquema esperado. "
        f"Columnas esperadas: {EXPECTED_COLUMNS}. "
        f"Columnas encontradas: {actual_columns}."
    )

df = df[EXPECTED_COLUMNS].copy()

df["texto_original"] = (
    df["descripcion"].fillna("").astype(str).str.strip()
    + " "
    + df["hashtags"].fillna("").astype(str).str.strip()
).str.replace(r"\s+", " ", regex=True).str.strip()

print(f"Archivo cargado desde: {DATA_PATH}")
print(f"Numero total de registros: {len(df)}")
display(df.head())


Archivo cargado desde: data\posts.csv
Numero total de registros: 5000


,id,descripcion,hashtags,categoria,texto_original
0,1,"GG, gráficos de locos con el ray tracing 🎮",#xbox,videojuegos,"GG, gráficos de locos con el ray tracing 🎮 #xbox"
1,2,Literalmente abriendo loot boxes y solo sale h...,#GGWP,videojuegos,Literalmente abriendo loot boxes y solo sale h...
2,3,marcando un hat-trick para salvar el partido s...,#Entreno #BeastMode #PR,deporte,marcando un hat-trick para salvar el partido s...
3,4,En bucle: desafinando en el coche a pleno pulm...,#ritmo,musica,En bucle: desafinando en el coche a pleno pulm...
4,5,NaN,#postres,cocina,#postres


## 2. Preparacion del texto

Para mantener comparabilidad con el notebook 01, se reutiliza la misma idea de preprocesamiento sobre `descripcion + hashtags`. Aunque los embeddings suelen tolerar texto mas natural que TF-IDF, conservar este paso permite trabajar exactamente con la misma senal textual ya preparada.


In [3]:
def preprocesar_texto(texto: str) -> str:
    texto = str(texto).lower()
    texto = texto.replace("#", " ")
    texto = re.sub(r"[^a-zA-Z0-9\sáéíóúüñ]", " ", texto)
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto


df["texto_limpio"] = df["texto_original"].apply(preprocesar_texto)
df_modelo = df[df["texto_limpio"].str.len() > 0].copy().reset_index(drop=True)

print(f"Registros originales: {len(df)}")
print(f"Registros con texto util: {len(df_modelo)}")
print(f"Registros excluidos por texto vacio: {len(df) - len(df_modelo)}")
display(df_modelo[["descripcion", "hashtags", "categoria", "texto_limpio"]].head(8))


Registros originales: 5000
Registros con texto util: 4938
Registros excluidos por texto vacio: 62


,descripcion,hashtags,categoria,texto_limpio
0,"GG, gráficos de locos con el ray tracing 🎮",#xbox,videojuegos,gg gráficos de locos con el ray tracing xbox
1,Literalmente abriendo loot boxes y solo sale h...,#GGWP,videojuegos,literalmente abriendo loot boxes y solo sale h...
2,marcando un hat-trick para salvar el partido s...,#Entreno #BeastMode #PR,deporte,marcando un hat trick para salvar el partido s...
3,En bucle: desafinando en el coche a pleno pulm...,#ritmo,musica,en bucle desafinando en el coche a pleno pulmó...
4,NaN,#postres,cocina,postres
5,Enamorado de este bicho: derrape guapo en la ú...,#Racing,coches,enamorado de este bicho derrape guapo en la úl...
6,"Increible, interior de cuero espectacular 🔥",#cochesclasicos #supercars #exhaust #gas,coches,increible interior de cuero espectacular coche...
7,Que ganas de esto: el agua esta helada pero me...,#Bronceado #Mar #Postu #Vacaciones #Vistas,playa,que ganas de esto el agua esta helada pero me ...


## 3. Generacion de embeddings

Un **embedding** es un vector numerico denso que intenta resumir el significado de un texto dentro de un espacio semantico. En ese espacio, dos frases cercanas suelen compartir tema o intencion, incluso si no usan exactamente las mismas palabras.

En este caso, cada publicacion se codifica con un modelo preentrenado de `SentenceTransformers`. El resultado es una matriz de dimensiones `n_posts x dimension_embedding`, donde cada fila representa un post.

In [4]:
modelo_embeddings = SentenceTransformer(EMBEDDING_MODEL_NAME)

embeddings_posts = modelo_embeddings.encode(
    df_modelo["texto_limpio"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

print(f"Modelo cargado: {EMBEDDING_MODEL_NAME}")
print(f"Matriz de embeddings: {embeddings_posts.shape[0]} posts x {embeddings_posts.shape[1]} dimensiones")
print("Primeras 5 componentes del primer embedding:")
print(np.round(embeddings_posts[0][:5], 4))


Batches: 100%|██████████| 78/78 [00:18<00:00,  4.18it/s]

Modelo cargado: all-MiniLM-L6-v2
Matriz de embeddings: 4938 posts x 384 dimensiones
Primeras 5 componentes del primer embedding:
[ 0.0229 -0.0354 -0.0238 -0.0645  0.0391]


## 4. Simulacion de un perfil de usuario

En un recomendador basado en contenido, el **perfil de usuario** es una representacion vectorial de los intereses del usuario. Aqui se simula un usuario que declara interes por varias categorias, por ejemplo `playa` y `viajes`.

Para construir su perfil, se genera un embedding para cada interes y despues se calcula el promedio. Ese promedio actua como un centro semantico de preferencias: cuanto mas cerca este un post de ese vector medio, mas afin se considera respecto al usuario.

In [5]:
embeddings_intereses = modelo_embeddings.encode(
    INTERESES_USUARIO,
    show_progress_bar=False,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

perfil_usuario = embeddings_intereses.mean(axis=0, keepdims=True)
perfil_usuario = perfil_usuario / np.linalg.norm(perfil_usuario, axis=1, keepdims=True)

print(f"Intereses del usuario: {INTERESES_USUARIO}")
print(f"Shape del perfil de usuario: {perfil_usuario.shape}")


Intereses del usuario: ['playa', 'viajes']
Shape del perfil de usuario: (1, 384)


## 5. Similitud coseno y recuperacion de recomendaciones

La **similitud coseno** mide cuanto se parecen dos vectores segun su orientacion. Un valor cercano a `1` indica gran parecido semantico; un valor cercano a `0` indica poca relacion. En recomendacion textual, esto permite ordenar los posts segun su proximidad al perfil del usuario.

No hay entrenamiento supervisado, no hay etiquetas objetivo y no hay division `train/test`: el sistema solo compara representaciones semanticas ya calculadas.

In [6]:
similitudes = cosine_similarity(perfil_usuario, embeddings_posts).ravel()

recomendaciones = df_modelo[["id", "descripcion", "hashtags", "categoria", "texto_limpio"]].copy()
recomendaciones["score_similitud"] = similitudes

top_k_posts = recomendaciones.sort_values("score_similitud", ascending=False).head(TOP_K).copy()
top_k_posts["score_similitud"] = top_k_posts["score_similitud"].round(4)

display(top_k_posts[["score_similitud", "categoria", "descripcion", "hashtags", "texto_limpio"]])


,score_similitud,categoria,descripcion,hashtags,texto_limpio
1516,0.8110,playa,NaN,#playa,playa
287,0.7162,playa,NaN,#playa#olas,playa olas
3941,0.5648,viajes,NaN,#Viajar,viajar
3261,0.5618,viajes,NaN,#viajar #paisaje #vuelos #aventura,viajar paisaje vuelos aventura
2831,0.5572,playa,NaN,#Olas #Verano #Atardecer #Playa #Costa,olas verano atardecer playa costa


In [8]:
for posicion, (_, fila) in enumerate(top_k_posts.iterrows(), start=1):
    print(f"Top {posicion} | score={fila['score_similitud']}")
    print(f"Categoria: {fila['categoria']}")
    print(f"Texto recomendado: {fila['texto_limpio']}")
    print("-" * 100)


Top 1 | score=0.8109999895095825
Categoria: playa
Texto recomendado: playa
----------------------------------------------------------------------------------------------------
Top 2 | score=0.7161999940872192
Categoria: playa
Texto recomendado: playa olas
----------------------------------------------------------------------------------------------------
Top 3 | score=0.5648000240325928
Categoria: viajes
Texto recomendado: viajar
----------------------------------------------------------------------------------------------------
Top 4 | score=0.5618000030517578
Categoria: viajes
Texto recomendado: viajar paisaje vuelos aventura
----------------------------------------------------------------------------------------------------
Top 5 | score=0.557200014591217
Categoria: playa
Texto recomendado: olas verano atardecer playa costa
----------------------------------------------------------------------------------------------------


## 6. Por que este enfoque es mas flexible que TF-IDF + LogisticRegression

El enfoque basado en embeddings es mas flexible por varias razones. Primero, no depende de entrenar un clasificador supervisado ni de disponer de etiquetas para cada nueva necesidad de recomendacion. Segundo, puede relacionar textos por significado semantico, aunque no compartan exactamente las mismas palabras. Tercero, produce un ranking continuo de afinidad en lugar de forzar una unica categoria como salida.

Frente al pipeline `TF-IDF + LogisticRegression`, este sistema puede adaptarse mejor a sinonimos, reformulaciones, nombres propios o expresiones nuevas que no estaban presentes en el vocabulario de entrenamiento. En otras palabras, mientras TF-IDF se apoya sobre todo en coincidencias lexicas, los embeddings permiten recomendar por cercania de significado, lo que resulta mas natural para un sistema de recomendacion basado en contenido.